# Once Upon a Matchmaker and a Streaming Service, in Python

In part one of the game, **Once Upon a Matchmaker**, four ladies and five gentlemen sign up with the Matchmaker,
a website that pairs ladies and gentlemen who like the same things. Every client answers the same three
questions, about dancing, reading and long walks. Turned into numbers, the answers are the clients'
**embeddings**: three numbers for each lady and three for each gentleman. The compatibility of a lady and a
gentleman is the inner product of their numbers, and all the compatibilities together make one table, the
product of two matrices: **matrix multiplication**.

In part two, **Once Upon a Streaming Service**, nobody fills in a form. The streaming service sees only ratings,
and looks for the numbers behind them: **matrix factorization**.

The notebook has two parts. Do each part when its game sends you here:

1. **Part 1, multiplying two matrices**, after part one, Once Upon a Matchmaker: one compatibility, then one
   lady's row, then the whole table (about 30 minutes).
2. **Part 2, the predictions and the losses**, in part two, Once Upon a Streaming Service, before the whole
   picture: the predictions of a guess, the loss of every rating, and their total (about 20 minutes).

Parts 3 and 4 go with the optional game, gradient descent up close, and are in a second notebook, also
optional: [the optional notebook in Colab](https://colab.research.google.com/github/eth-ainit-hs26/w2-cx-public/blob/main/02_cx_matrix_factorization_optional_student.ipynb).

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A
folded cell with a title holds plumbing: run it, and open it only if you are curious. The 💡 hints are folded
cells too: double-click one to open it, and open the next one only if you are still stuck. After each task, a
check cell tells you whether your code works.

# Part 1. Multiplying two matrices

In part one of the game the Matchmaker filled its whole table of compatibilities from the clients' answers, one
inner product for each lady and each gentleman, and the operation got its name: **matrix multiplication**. Now
you teach it to a computer, in small steps:

- **Task A**: one compatibility, Rose and Felix;
- **Task B**: one row, a lady's compatibility with every gentleman;
- **Task C**: the whole table, which is matrix multiplication;
- **Task D**: the rule that says when two matrices can be multiplied.

At the end your code multiplies the bigger pair of the game. Run the setup cell first.

In [ ]:
#@title Setup of part 1 (run this first) { display-mode: "form" }
# The plumbing of part 1: show_matrix, which draws a matrix as a table, and the checks of tasks A to D.
# You do not need to read it. A check prints one short message and never stops the notebook.
try:
    from IPython import get_ipython as _p1_get_ipython
    from IPython.display import display as _p1_display
except ImportError:
    _p1_get_ipython = None
    _p1_display = None

# The game's numbers, kept here as well, so that the checks still work if you change the lists in the notebook
_P1_LADIES = [[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]]
_P1_GENTLEMEN = [[1, -2, 1], [-2, 2, 1], [-2, 2, -2], [2, -2, -2], [-1, 0, 2]]   # as on the form: one row per gentleman
_P1_TURNED = [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]   # transposed: one row per question, one column per gentleman
_P1_LADY_NAMES = ['Clara', 'Eliza', 'Harriet', 'Rose']
_P1_GENTLEMAN_NAMES = ['Arthur', 'Felix', 'George', 'Oscar', 'Walter']
_P1_QUESTIONS = ['dancing', 'reading', 'long walks']
_P1_BIG_A = [[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]]
_P1_BIG_B = [[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]]
_P1_BIG_PRODUCT = [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]]
_P1_ONE = (3, 1)   # task A: Rose's row in ladies_matrix, and Felix's column in gentlemen_transposed


def _p1_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p1_column(matrix, j):
    numbers = []
    for row in matrix:
        numbers.append(row[j])
    return numbers


def _p1_columns(matrix):
    """Every column of matrix, each as a list, or [] when matrix is not a table."""
    try:
        return [_p1_column(matrix, j) for j in range(len(matrix[0]))]
    except Exception:
        return []


def _p1_product(matrix_a, matrix_b):
    """Every row of matrix_a with every column of matrix_b."""
    product = []
    for row_a in matrix_a:
        new_row = []
        for j in range(len(matrix_b[0])):
            new_row.append(_p1_inner(row_a, _p1_column(matrix_b, j)))
        product.append(new_row)
    return product


def _p1_try(make):
    """make(), or None when it stops with an error."""
    try:
        return make()
    except Exception:
        return None


def _p1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p1_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _p1_count(n, word):
    """1 row, 3 rows"""
    if n == 1:
        return "1 " + word
    return "%d %ss" % (n, word)


def _p1_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


# show_matrix: a matrix as a small table

def _p1_entry(x):
    """One entry of a table as text: 7, -2, 0.75 (two decimals for a decimal number), ? for None."""
    if x is None:
        return "?"
    if x is Ellipsis:
        return "..."
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, float):
        return "%.2f" % x
    return str(x)


def _p1_tint(x):
    """The background of an entry, as in the game: blue above 0, orange below 0, grey for 0, and a stronger
    tint from 2 and -2 on. The tints are see-through, so the table reads in a light and in a dark theme."""
    if not _p1_is_number(x):
        return ""
    if x > 0:
        colour = "59,130,214"
    elif x < 0:
        colour = "224,116,42"
    else:
        return "background:rgba(140,140,140,0.16);"
    strength = 0.16
    if abs(x) >= 2:
        strength = 0.32
    return "background:rgba(%s,%.2f);" % (colour, strength)


def _p1_not_a_table(matrix):
    """None when matrix is a table (a list of rows of the same length, each entry a single value), else a pair
    (kind, reason): kind "wait" when a task is not finished yet, "shape" when the value has another shape."""
    if matrix is None:
        return ("wait", "there is no table yet, only None (does the function end with a return line?)")
    if matrix is Ellipsis:
        return ("wait", "the table is still ..., a gap in a 🎯 cell")
    if not isinstance(matrix, (list, tuple)):
        return ("shape", "this is a single value, %s, and a table is a list of rows" % _p1_short(matrix))
    if len(matrix) == 0:
        return ("wait", "the table is an empty list, []")
    for r in range(len(matrix)):
        row = matrix[r]
        if row is Ellipsis:
            return ("wait", "row %d of the table is still ..., a gap in a 🎯 cell" % r)
        if row is None:
            return ("wait", "row %d of the table is None (does the function that makes a row end with a return "
                            "line?)" % r)
        if not isinstance(row, (list, tuple)):
            return ("shape", "row %d is %s, a single value, and each row of a table is a list of numbers"
                    % (r, _p1_short(row)))
    lengths = []
    for row in matrix:
        lengths.append(len(row))
    if max(lengths) == 0:
        if len(matrix) == 1:
            return ("wait", "its only row is an empty list, []")
        return ("wait", "its rows are empty lists, []")
    if min(lengths) != max(lengths):
        return ("shape", "its rows have different lengths, %s, and the rows of a table are all equally long"
                % _p1_and(lengths))
    for row in matrix:
        for x in row:
            if x is Ellipsis:
                return ("wait", "it still holds a ..., a gap in a 🎯 cell")
            if isinstance(x, (list, tuple)):
                return ("shape", "its entries are lists, and each entry of a table is a single number")
    return None


def _p1_labels(names, n):
    """The labels of n rows or columns: the names when there is one name for each, the numbers from 0 when
    names is None, and None when the names do not fit."""
    if names is None:
        return [str(i) for i in range(n)]
    if isinstance(names, (list, tuple)) and len(names) == n:
        return [str(x) for x in names]
    return None


def show_matrix(matrix, row_names=None, column_names=None):
    """Show a matrix (a list of rows) as a table, with a name for each row and each column. Without names,
    the rows and the columns are numbered from 0, the way Python counts. When matrix is not a table yet
    (None, a gap, an empty list, rows of different lengths), it prints one calm line and stops nothing."""
    try:
        problem = _p1_not_a_table(matrix)
        if problem is not None:
            kind, reason = problem
            if kind == "wait":
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % reason)
            else:
                print("⏳ This is not a table yet: %s. The check of your function says more." % reason)
            return
        rows = len(matrix)
        columns = len(matrix[0])
        size = "%s and %s" % (_p1_count(rows, "row"), _p1_count(columns, "column"))
        row_labels = _p1_labels(row_names, rows)
        column_labels = _p1_labels(column_names, columns)
        if row_labels is None or column_labels is None:
            swapped = (isinstance(row_names, (list, tuple)) and isinstance(column_names, (list, tuple))
                       and len(row_names) == columns and len(column_names) == rows and rows != columns)
            if swapped:
                print("Note: this table has %s, and the names are for %s and %s. Are its rows and columns "
                      "swapped? They are numbered below." % (size, _p1_count(columns, "row"),
                                                              _p1_count(rows, "column")))
            else:
                print("Note: the names do not fit this table of %s, so it is numbered below." % size)
            row_labels = [str(i) for i in range(rows)]
            column_labels = [str(j) for j in range(columns)]

        entries = []
        for r in range(rows):
            line = []
            for c in range(columns):
                line.append(_p1_entry(matrix[r][c]))
            entries.append(line)

        # Plain text, for screens that cannot show the table
        name_width = max(len(label) for label in row_labels)
        widths = []
        for c in range(columns):
            width = len(column_labels[c])
            for r in range(rows):
                width = max(width, len(entries[r][c]))
            widths.append(width)
        lines = [" " * name_width + "".join("   " + column_labels[c].rjust(widths[c]) for c in range(columns))]
        for r in range(rows):
            lines.append(row_labels[r].ljust(name_width)
                         + "".join("   " + entries[r][c].rjust(widths[c]) for c in range(columns)))
        lines.append(size)
        text = "\n".join(line.rstrip() for line in lines)

        # The same table in HTML, each number tinted by its sign
        from html import escape
        edge = "border:1px solid rgba(128,128,128,0.3);"
        html = '<table style="border-collapse:collapse;font-size:15px;margin:6px 0 2px 0">'
        html += '<tr><th style="padding:4px 8px"></th>'
        for c in range(columns):
            html += '<th style="padding:4px 10px;text-align:center;font-weight:600">%s</th>' % escape(column_labels[c])
        html += "</tr>"
        for r in range(rows):
            html += '<tr><th style="padding:4px 12px 4px 4px;text-align:left;font-weight:600">%s</th>' % escape(
                row_labels[r])
            for c in range(columns):
                html += ('<td style="padding:5px 12px;min-width:2.2em;text-align:center;'
                         'font-variant-numeric:tabular-nums;%s%s">%s</td>'
                         % (edge, _p1_tint(matrix[r][c]), escape(entries[r][c])))
            html += "</tr>"
        html += "</table>"
        html += '<div style="font-size:13px;opacity:0.75;margin:0 0 8px 2px">%s</div>' % escape(size)

        shell = None
        if _p1_get_ipython is not None:
            shell = _p1_get_ipython()
        if _p1_display is None or shell is None:
            print(text)
        else:
            _p1_display({"text/plain": text, "text/html": html}, raw=True)
    except Exception as error:
        print("The table could not be shown (%s). Run the setup cell of part 1 again, then this cell." % error)


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P1_TESTS = {
    "inner_product": [
        (([2, -1, -2], [1, -2, 1]), 2,
         '',
         ''),
        (([-2, 2, 2], [2, -2, -2]), -12,
         '',
         ''),
        (([1, -1, 2], [1, -2, 1]), 5,
         '',
         ''),
        (([1, -2, 0, 2], [1, 1, 0, -1]), -3,
         '',
         ''),
        (([2], [-2]), -4,
         '',
         ''),
    ],
    "column": [
        (([[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]], 0), [1, -2, 1],
         '',
         ''),
        (([[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]], 1), [-2, 2, 1],
         '',
         ''),
        (([[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]], 5), [0, -1, 1, -2],
         '',
         ''),
        (([[1, 2], [3, 4], [5, 6]], 1), [2, 4, 6],
         '',
         ''),
    ],
    "transpose": [
        (([[1, -2, 1], [-2, 2, 1], [-2, 2, -2], [2, -2, -2], [-1, 0, 2]],), [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]],
         '',
         ''),
        (([[1, 2, 3]],), [[1], [2], [3]],
         '',
         ''),
        (([[1, 2], [3, 4], [5, 6]],), [[1, 3, 5], [2, 4, 6]],
         '',
         ''),
    ],
    "compatibilities_for_lady": [
        (([2, -1, -2], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [2, -8, -2, 10, -6],
         'compatibilities_for_lady(ladies_matrix[0], gentlemen_transposed)',
         "Clara's compatibility with each gentleman, in the order of the columns of gentlemen_transposed"),
        (([1, -1, 2], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [5, -2, -8, 0, 3],
         'compatibilities_for_lady(ladies_matrix[1], gentlemen_transposed)',
         "Eliza's compatibility with each gentleman, in the order of the columns of gentlemen_transposed"),
        (([-2, 1, -2], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [-6, 4, 10, -2, -2],
         'compatibilities_for_lady(ladies_matrix[2], gentlemen_transposed)',
         "Harriet's compatibility with each gentleman, in the order of the columns of gentlemen_transposed"),
        (([-2, 2, 2], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [-4, 10, 4, -12, 6],
         'compatibilities_for_lady(ladies_matrix[3], gentlemen_transposed)',
         "Rose's compatibility with each gentleman, in the order of the columns of gentlemen_transposed"),
        (([2, -1, -2], [[2, 1], [-2, -2], [-2, 1]]), [10, 2],
         'compatibilities_for_lady([2, -1, -2], [[2, 1], [-2, -2], [-2, 1]])',
         'Clara with two gentlemen only, Oscar in the first column and Arthur in the second: one compatibility for each, in that order'),
        (([-2, 2, 2], [[-2], [2], [1]]), [10],
         'compatibilities_for_lady([-2, 2, 2], [[-2], [2], [1]])',
         'With one gentleman, one column, the answer is a list that holds one compatibility'),
        (([1, -2, 0, 2], [[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]]), [-3, 6, 4, -3, 1, -2],
         'compatibilities_for_lady(big_a[0], big_b)',
         'big_a and big_b are the bigger pair of the game (you meet them at the end of part 1): a row of 4 numbers and a matrix of 6 columns give 6 inner products'),
    ],
    "matrix_multiply": [
        (([[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [[2, -8, -2, 10, -6], [5, -2, -8, 0, 3], [-6, 4, 10, -2, -2], [-4, 10, 4, -12, 6]],
         'matrix_multiply(ladies_matrix, gentlemen_transposed)',
         "The Matchmaker's table: one row per lady, one column per gentleman"),
        (([[2, -1, -2]], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), [[2, -8, -2, 10, -6]],
         'matrix_multiply([[2, -1, -2]], gentlemen_transposed)',
         "A matrix with one row, Clara's, times the gentlemen: a table with one row"),
        (([[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]], [[1], [-2], [1]]), [[2], [5], [-6], [-4]],
         'matrix_multiply(ladies_matrix, [[1], [-2], [1]])',
         "The ladies times one column, Arthur's: a table with one column"),
        (([[3]], [[-2]]), [[-6]],
         'matrix_multiply([[3]], [[-2]])',
         'Two matrices of one row and one column: a table of one row that holds one number'),
        (([[1, 2]], [[3, 4, 5], [6, 7, 8]]), [[15, 18, 21]],
         'matrix_multiply([[1, 2]], [[3, 4, 5], [6, 7, 8]])',
         '1 row and 2 columns times 2 rows and 3 columns: a table of 1 row and 3 columns'),
        (([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]],
         'matrix_multiply([[1, 2], [3, 4]], [[5, 6], [7, 8]])',
         'Two matrices of 2 rows and 2 columns: every row of the first with every column of the second'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]]), [[-3, 6, 4, -3, 1, -2], [3, -4, 5, -2, 6, -2], [-2, 5, -3, 3, -3, 0], [4, -4, -3, 3, -2, 3], [0, 5, 4, -2, 0, 0]],
         'matrix_multiply(big_a, big_b)',
         'The bigger pair of the game (you meet it at the end of part 1)'),
    ],
    "can_multiply": [
        (([[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]], [[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]]), True,
         'can_multiply(ladies_matrix, gentlemen_transposed)',
         'ladies_matrix has 3 columns, one per question, and gentlemen_transposed 3 rows, one per question'),
        (([[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]], [[1, -2, 1], [-2, 2, 1], [-2, 2, -2], [2, -2, -2], [-1, 0, 2]]), False,
         'can_multiply(ladies_matrix, gentlemen_matrix)',
         'ladies_matrix has 3 columns, and gentlemen_matrix, the table before transposing, 5 rows, one per gentleman'),
        (([[1, -2, 0, 2], [2, 1, -1, 0], [-1, 0, 2, 1], [0, 2, 1, -2], [2, -1, 1, 1]], [[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]]), True,
         'can_multiply(big_a, big_b)',
         '4 columns and 4 rows: the bigger pair of the game, which you meet at the end of part 1'),
        (([[1, 0], [0, 1], [1, 1]], [[1, 2, 0, -1], [0, 1, 1, 2]]), True,
         'can_multiply([[1, 0], [0, 1], [1, 1]], [[1, 2, 0, -1], [0, 1, 1, 2]])',
         '2 columns and 2 rows: a row of the first and a column of the second both hold 2 numbers'),
        (([[1, 2]], [[3, 4], [5, 6], [7, 8]]), False,
         'can_multiply([[1, 2]], [[3, 4], [5, 6], [7, 8]])',
         '2 columns and 3 rows, although both matrices have 2 columns'),
        (([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]]), False,
         'can_multiply([[1, 2, 3], [4, 5, 6]], [[1, 2], [3, 4]])',
         '3 columns and 2 rows, although both matrices have 2 rows'),
        (([[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]], [[1, 0, 2, -1, 1, 0], [1, -2, 0, 1, 2, -1], [0, 2, -1, 1, -2, 1], [-1, 1, 1, 0, 2, -2]]), False,
         'can_multiply(ladies_matrix, big_b)',
         '3 columns and 4 rows'),
        (([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]]), True,
         'can_multiply([[1, 2, 0]], [[1, 0], [0, 1], [2, 2]])',
         '3 columns and 3 rows, although the first matrix has 1 row and the second 2 columns'),
        (([[1, -2, -2, 2, -1], [-2, 2, 2, -2, 0], [1, 1, -2, -2, 2]], [[2, -1, -2], [1, -1, 2], [-2, 1, -2], [-2, 2, 2]]), False,
         'can_multiply(gentlemen_transposed, ladies_matrix)',
         'The other way round: gentlemen_transposed has 5 columns and ladies_matrix 4 rows'),
    ],
}

_P1_HINTS = {'compatibilities_for_lady': 'in the loop, compatibility = inner_product(lady_row, gentleman_column), then compatibilities.append(compatibility).', 'matrix_multiply': 'row i of the product is compatibilities_for_lady(matrix_a[i], matrix_b): the inner products of matrix_a[i] with every column of matrix_b.', 'can_multiply': 'rows_b = len(matrix_b); then, if columns_a == rows_b, return True, else return False.'}
_P1_SUCCESS = {'one_compatibility': "Task A works: Rose's row [-2, 2, 2] and Felix's column [-2, 2, 1] give (-2)*(-2) + 2*2 + 2*1 = 4 + 4 + 2 = 10. Felix is her best match.", 'compatibilities_for_lady': "Clara's compatibilities with Arthur, Felix, George, Oscar and Walter are [2, -8, -2, 10, -6].", 'matrix_multiply': "It gives the Matchmaker's table of compatibilities, 4 rows and 5 columns, and the bigger product of the game, 5 rows and 6 columns.", 'can_multiply': "True for the ladies and the gentlemen transposed, 3 columns and 3 rows; False for the gentlemen's table as on the form, 3 columns and 5 rows.", 'big_product': "big_product has 5 rows and 6 columns and the same 30 numbers as the game's bigger product."}
_P1_COUNTING = {'ladies_matrix': "Python counts from 0: ladies_matrix[0] is Clara's row, ladies_matrix[1] Eliza's, ladies_matrix[2] Harriet's and ladies_matrix[3] Rose's.", 'gentlemen_transposed': "Python counts from 0: column(gentlemen_transposed, 0) is Arthur's column, column(gentlemen_transposed, 1) Felix's, column(gentlemen_transposed, 2) George's, column(gentlemen_transposed, 3) Oscar's and column(gentlemen_transposed, 4) Walter's."}
_P1_GIVEN = ("inner_product", "column", "transpose")   # the three functions of the given cell
_P1_RUN_GIVEN = "Run the cell of inner_product, column and transpose again (it is given, so undo any change to it)"


def _p1_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error), where error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _p1_right(name, answer, expected):
    """True when answer is the expected one; can_multiply must answer True or False themselves."""
    try:
        if name == "can_multiply":
            return isinstance(answer, bool) and answer == expected
        return bool(answer == expected)
    except Exception:
        return False


def _p1_first_failure(name, fn):
    """The first test of name that fn fails, or None when fn passes them all."""
    for inputs, expected, call, why in _P1_TESTS[name]:
        answer, error = _p1_call(fn, inputs)
        if error is not None or not _p1_right(name, answer, expected):
            return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                    "why": why}
    return None


def _p1_given_broken():
    """The name of a given function that a participant changed or deleted, or None when all three work."""
    for name in _P1_GIVEN:
        fn = globals().get(name)
        if not callable(fn) or _p1_first_failure(name, fn) is not None:
            return name
    return None


def _p1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function, so no source is needed; a gap that stands on
    # its own line is only visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p1_rows_of(value):
    """True when value is a list of lists."""
    return isinstance(value, list) and len(value) > 0 and isinstance(value[0], list)


def _p1_explain_row(failure):
    """(what went wrong, hint) for compatibilities_for_lady."""
    answer, expected = failure["answer"], failure["expected"]
    lady_row, gentlemen = failure["inputs"]
    error = failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "an index went past the end of a list. j counts the gentlemen, the columns of "
                          "gentlemen_transposed, and column(gentlemen_transposed, j) is the column of gentleman j. "
                          "gentlemen_transposed[j] would be row j, every gentleman's answer to one question, and "
                          "gentlemen_transposed has %s." % _p1_count(len(gentlemen), "row"))
        if "TypeError" in error:
            return (None, "inner_product takes two lists of numbers: lady_row and gentleman_column, the column of "
                          "one gentleman. gentlemen_transposed alone is the whole matrix.")
        if "NameError" in error and "column" in error:
            return (None, "column is given: run the cell of inner_product, column and transpose, then the cell of "
                          "your function, then this check.")
        return (None, _P1_HINTS["compatibilities_for_lady"])
    if answer is None:
        return (None, "end the function with return compatibilities.")
    if not isinstance(answer, list):
        return ("gave %s, expected the list %r" % (_p1_short(answer), expected),
                "return the list compatibilities, with one compatibility per gentleman. A single number is the "
                "compatibility with one gentleman only.")
    if len(answer) == 0:
        return ("gave an empty list, expected %r" % expected,
                "compatibilities is still empty: in each round of the loop, put the compatibility at the end of the "
                "list with compatibilities.append(compatibility).")
    for x in answer:
        if isinstance(x, (list, tuple)):
            return ("gave a list of lists, %s, expected %r" % (_p1_short(answer), expected),
                    "each entry of compatibilities is one number. Did you append the column of a gentleman, "
                    "gentleman_column, in place of the compatibility?")
    # The rule of 2026-11-03: the lady's row with every row of gentlemen_transposed
    by_rows = _p1_try(lambda: [_p1_inner(lady_row, row) for row in gentlemen])
    if by_rows is not None and answer == by_rows and by_rows != expected:
        return ("gave %r, expected %r" % (answer, expected),
                "these pair lady_row with the rows of gentlemen_transposed, and a row of gentlemen_transposed holds "
                "every gentleman's answer to one question. A gentleman's numbers stand in a column: "
                "gentleman_column = column(gentlemen_transposed, j), with j counting the columns, "
                "range(len(gentlemen_transposed[0])).")
    if len(answer) != len(expected):
        return ("gave %s, %s, expected %d, %r" % (_p1_count(len(answer), "compatibility").replace("ys", "ies"),
                                                  _p1_short(answer), len(expected), expected),
                "one compatibility per column of gentlemen_transposed, one per gentleman: append exactly one "
                "compatibility in each round of the loop, inside the loop.")
    if len(answer) > 1 and answer.count(answer[0]) == len(answer) and expected.count(expected[0]) != len(expected):
        return ("gave %r, expected %r" % (answer, expected),
                "every compatibility is the same: in round j of the loop, use the column of gentleman j, "
                "column(gentlemen_transposed, j).")
    first = _p1_try(lambda: [_p1_inner(_P1_LADIES[0], _p1_column(gentlemen, j)) for j in range(len(gentlemen[0]))])
    if first is not None and answer == first and lady_row != _P1_LADIES[0]:
        return ("gave %r, expected %r" % (answer, expected),
                "these are %s's compatibilities: use lady_row, the row that the function gets, in place of "
                "ladies_matrix[0]." % _P1_LADY_NAMES[0])
    return ("gave %r, expected %r" % (answer, expected), _P1_HINTS["compatibilities_for_lady"])


def _p1_explain_product(answer, matrix_a, matrix_b, expected):
    """(what went wrong, hint) for a product that differs from the expected one. answer is not None."""
    rows, columns = len(expected), len(expected[0])
    wanted = "%s, one for each row of matrix_a, and %s, one for each column of matrix_b" % (
        _p1_count(rows, "row"), _p1_count(columns, "column"))
    if not isinstance(answer, list):
        return ("gave %s; expected a table of %s" % (_p1_short(answer), wanted),
                "return product, the list of rows.")
    if len(answer) == 0:
        return ("gave an empty list; expected a table of %s" % wanted,
                "product is still empty: append each new_row to product, inside the loop.")
    flat = True
    for row in answer:
        if not _p1_is_number(row):
            flat = False
    if flat:
        return ("gave a list of %s, %s; expected a table of %s" % (_p1_count(len(answer), "number"),
                                                                   _p1_short(answer), wanted),
                "the product is a list of rows. new_row is a whole row, the inner products of matrix_a[i] with every "
                "column of matrix_b, and product.append(new_row) adds it to the table.")
    for r in range(len(answer)):
        row = answer[r]
        if row is None:
            return ("gave a table whose row %d is None" % r,
                    "new_row is None: does compatibilities_for_lady end with return compatibilities?")
        if not isinstance(row, list):
            return ("gave a table whose row %d is %s, a single value" % (r, _p1_short(row)),
                    "each row of the product is a list of numbers, one for each column of matrix_b.")
    for row in answer:
        for x in row:
            if isinstance(x, (list, tuple)):
                return ("gave a table whose entries are lists, such as %s" % _p1_short(x),
                        "each entry of the product is one number, an inner product. Put numbers into new_row: "
                        "appending a whole list puts a list inside the row.")
    lengths = []
    for row in answer:
        lengths.append(len(row))
    if max(lengths) == 0:
        return ("gave %s that are all empty lists; expected %s" % (_p1_count(len(answer), "row"), wanted),
                "new_row stays empty: in the loop over the columns of matrix_b, put each inner product at the end "
                "of new_row with new_row.append(...).")
    got = "a table of %s and %s" % (_p1_count(len(answer), "row"), _p1_count(max(lengths), "column"))
    # The Matchmaker's table again, although the function got other matrices: it reads the notebook's lists
    if (matrix_a, matrix_b) != (_P1_LADIES, _P1_TURNED):
        if answer == _p1_product(_P1_LADIES, _P1_TURNED):
            return ("gave the Matchmaker's table of compatibilities again; expected a table of %s" % wanted,
                    "use matrix_a and matrix_b, the two matrices that the function gets, in place of "
                    "ladies_matrix and gentlemen_transposed.")
    # The rule of 2026-11-03: every row of matrix_a with every row of matrix_b
    by_rows = _p1_try(lambda: [[_p1_inner(row_a, row_b) for row_b in matrix_b] for row_a in matrix_a])
    if by_rows is not None and answer == by_rows and by_rows != expected:
        return ("gave %s, whose entries pair rows with rows; expected %s" % (got, wanted),
                "this pairs every row of matrix_a with every row of matrix_b, as if matrix_b were the gentlemen's "
                "table before transposing. A row of matrix_a meets a column of matrix_b: the entry in row i and "
                "column j is inner_product(matrix_a[i], column(matrix_b, j)), and "
                "compatibilities_for_lady(matrix_a[i], matrix_b) gives the whole row i.")
    growing = []
    for r in range(len(answer)):
        growing.append(columns * (r + 1))
    one_list = True
    for row in answer:
        if row is not answer[0]:
            one_list = False
    if len(answer) > 1 and (lengths == growing or (one_list and lengths[0] != columns)):
        return ("gave rows of %s numbers; expected %s" % (_p1_and(lengths), wanted),
                "new_row keeps growing: start a new, empty row, new_row = [], for each row of matrix_a, inside "
                "the outer loop.")
    if min(lengths) != max(lengths):
        return ("gave rows of different lengths, %s; expected %s" % (_p1_and(lengths), wanted),
                "every row of the product has one number for each column of matrix_b.")
    got_rows, got_columns = len(answer), lengths[0]
    if (got_rows, got_columns) != (rows, columns):
        if (got_rows, got_columns) == (columns, rows):
            return ("gave %s; expected %s" % (got, wanted),
                    "rows and columns are swapped. The outer loop goes over the rows of matrix_a, and each new "
                    "row holds one inner product for each column of matrix_b.")
        if got_rows != rows:
            return ("gave %s; expected %s" % (got, wanted),
                    "the product has one row for each row of matrix_a: append one new_row in each round of the "
                    "loop over the rows of matrix_a.")
        if got_columns == len(matrix_b):
            return ("gave %s; expected %s" % (got, wanted),
                    "each row holds one number for each row of matrix_b, len(matrix_b). The product has one "
                    "column for each column of matrix_b: the loop over j runs over range(len(matrix_b[0])).")
        return ("gave %s; expected %s" % (got, wanted),
                "each row of the product holds one number for each column of matrix_b: the inner products of "
                "matrix_a[i] with column(matrix_b, 0), column(matrix_b, 1) and so on.")
    # The right size, some wrong numbers
    flat_rows = True
    for r in range(rows):
        if answer[r].count(answer[r][0]) != columns:
            flat_rows = False
    expected_flat = True
    for r in range(rows):
        if expected[r].count(expected[r][0]) != columns:
            expected_flat = False
    if columns > 1 and flat_rows and not expected_flat:
        return ("has the right size, %s and %s, but every number in a row is the same: row 0 is %r, expected %r"
                % (_p1_count(rows, "row"), _p1_count(columns, "column"), answer[0], expected[0]),
                "the entry in row i and column j pairs matrix_a[i] with column j of matrix_b, column(matrix_b, j), "
                "a different column for each j. Check which index goes with which matrix.")
    for i in range(rows):
        for j in range(columns):
            if answer[i][j] != expected[i][j]:
                return ("has the right size, %s and %s, but the entry in row %d and column %d is %s, expected %r"
                        % (_p1_count(rows, "row"), _p1_count(columns, "column"), i, j, _p1_short(answer[i][j]),
                           expected[i][j]),
                        "the entry in row %d and column %d is the inner product of matrix_a[%d], %r, with column %d "
                        "of matrix_b, %r." % (i, j, i, matrix_a[i], j, _p1_column(matrix_b, j)))
    return ("gave %s" % _p1_short(answer), _P1_HINTS["matrix_multiply"])


def _p1_explain_columns(fn, failure):
    """(what went wrong, hint) for can_multiply, from the pattern of its answers on all the tests."""
    answer, error = failure["answer"], failure["error"]
    if error is not None:
        if "IndexError" in error:
            return (None, "the columns of matrix_a are the length of its first row, len(matrix_a[0]), and the rows "
                          "of matrix_b the length of matrix_b itself, len(matrix_b). Every matrix has a first row, "
                          "even a matrix with a single row.")
        return (None, "len gives the length of a list: len(matrix_b) is the number of rows of matrix_b, and "
                      "len(matrix_a[0]) the number of columns of matrix_a.")
    if answer is None:
        return (None, "end with a return line: if columns_a == rows_b, return True, else return False.")
    if not isinstance(answer, bool):
        if answer in ("True", "False"):
            return ("gave the text %r, expected %r" % (answer, failure["expected"]),
                    "return True or False without quotes: they are Python's own values for yes and no.")
        return ("gave %s, expected %r" % (_p1_short(answer), failure["expected"]),
                "return True or False: compare the two numbers with == inside an if.")
    answers, expected, same_columns, same_rows, rows_columns = [], [], [], [], []
    for inputs, wanted, call, why in _P1_TESTS["can_multiply"]:
        matrix_a, matrix_b = inputs
        answers.append(_p1_call(fn, inputs)[0])
        expected.append(wanted)
        same_columns.append(len(matrix_a[0]) == len(matrix_b[0]))
        same_rows.append(len(matrix_a) == len(matrix_b))
        rows_columns.append(len(matrix_a) == len(matrix_b[0]))
    what = "gave %r, expected %r" % (answer, failure["expected"])
    if answers == same_columns:
        return (what, "you compared the columns of both matrices, len(matrix_a[0]) and len(matrix_b[0]). That says "
                      "True for the ladies with the gentlemen's table before transposing. A row of matrix_a meets a "
                      "column of matrix_b, so the columns of matrix_a must match the rows of matrix_b: compare "
                      "len(matrix_a[0]) with len(matrix_b).")
    if answers == same_rows:
        return (what, "you compared the numbers of rows, len(matrix_a) and len(matrix_b). Compare the columns of "
                      "matrix_a, len(matrix_a[0]), with the rows of matrix_b, len(matrix_b).")
    if answers == rows_columns:
        return (what, "you compared the rows of matrix_a, len(matrix_a), with the columns of matrix_b, "
                      "len(matrix_b[0]), the other way round. Compare the columns of matrix_a, len(matrix_a[0]), "
                      "with the rows of matrix_b, len(matrix_b).")
    if answers == [True] * len(answers):
        return (what, "your function says True every time, also for a pair that cannot be multiplied. Return "
                      "False when columns_a and rows_b differ (else: return False).")
    if answers == [False] * len(answers):
        return (what, "your function never says True. Return True when columns_a == rows_b.")
    flipped = []
    for x in expected:
        flipped.append(not x)
    if answers == flipped:
        return (what, "True and False are swapped: return True when the columns of matrix_a match the rows of "
                      "matrix_b.")
    return (what, _P1_HINTS["can_multiply"])


def _p1_explain(name, fn, failure):
    """(what went wrong, hint) for the first test that fn fails."""
    if name == "compatibilities_for_lady":
        return _p1_explain_row(failure)
    if name == "can_multiply":
        return _p1_explain_columns(fn, failure)
    if failure["error"] is not None:
        error = failure["error"]
        if "IndexError" in error:
            return (None, "an index went past the end of a matrix. i counts the rows of matrix_a, so row i is "
                          "matrix_a[i]; j counts the columns of matrix_b, range(len(matrix_b[0])), so column j is "
                          "column(matrix_b, j). Pair matrix_a[i] with column(matrix_b, j).")
        if "TypeError" in error:
            return (None, "inner_product takes two lists of numbers, such as matrix_a[i] and column(matrix_b, j); "
                          "compatibilities_for_lady takes one row and a whole matrix, such as matrix_a[i] and "
                          "matrix_b.")
        if "NameError" in error and "column" in error:
            return (None, "column is given: run the cell of inner_product, column and transpose, then the cell of "
                          "your function, then this check.")
        return (None, _P1_HINTS["matrix_multiply"])
    if failure["answer"] is None:
        return (None, "end the function with return product.")
    matrix_a, matrix_b = failure["inputs"]
    return _p1_explain_product(failure["answer"], matrix_a, matrix_b, failure["expected"])


def _p1_check(name, fn, needs):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    failure = _p1_first_failure(name, fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P1_TESTS[name]), _P1_SUCCESS[name]))
        return
    not_finished = ("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, "
                    "run its cell, then run this check again." % name)
    if _p1_has_gap(fn):
        print(not_finished)
        return
    for other, place in needs:
        if not _p1_uses(fn, other):
            continue
        other_fn = globals().get(other)
        if other_fn is None or _p1_first_failure(other, other_fn) is not None:
            if other in _P1_GIVEN:
                print("⏳ %s uses %s, which does not work any more. %s, then run this check again."
                      % (name, other, _P1_RUN_GIVEN))
            else:
                print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, "
                      "then run this check again." % (name, other, place, place))
            return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    what, hint = _p1_explain(name, fn, failure)
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None). Does the function end with a return line?" % failure["call"])
    else:
        print("❌ %s %s. %s." % (failure["call"], what, failure["why"]))
    print("   💡 Hint: " + hint)


def check_compatibilities_for_lady(fn):
    """Task B: tests compatibilities_for_lady on the four ladies, on fewer gentlemen and on the bigger pair."""
    try:
        _p1_check("compatibilities_for_lady", fn, [("inner_product", "the given cell"), ("column", "the given cell")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_matrix_multiply(fn):
    """Task C: tests matrix_multiply on seven pairs of matrices, from 1 x 1 to the bigger pair of the game."""
    try:
        _p1_check("matrix_multiply", fn, [("inner_product", "the given cell"), ("column", "the given cell"),
                                          ("compatibilities_for_lady", "task B")])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def check_can_multiply(fn):
    """Task D: tests can_multiply on nine pairs of matrices, some that fit and some that do not."""
    try:
        _p1_check("can_multiply", fn, [])
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_which(value, lists):
    """The index of value among lists, or -1."""
    for i in range(len(lists)):
        try:
            if value == lists[i]:
                return i
        except Exception:
            pass
    return -1


def _p1_row_problem(value):
    """None when value is Rose's row of ladies_matrix, else (what went wrong, hint), for task A."""
    i = _P1_ONE[0]
    matrix, names = _P1_LADIES, _P1_LADY_NAMES
    if _p1_which(value, [matrix[i]]) == 0:
        return None
    k = _p1_which(value, matrix)
    if k >= 0:
        return ("rose_row is %s, the row of %s, ladies_matrix[%d]" % (_p1_short(value), names[k], k),
                _P1_COUNTING["ladies_matrix"])
    k = _p1_which(value, _p1_columns(matrix))
    if k >= 0:
        return ("rose_row is %s, column %d of ladies_matrix: every lady's answer about %s"
                % (_p1_short(value), k, _P1_QUESTIONS[k]),
                "a lady's numbers stand in a row: ladies_matrix[%d] is %s's row." % (i, names[i]))
    if _p1_which(value, _P1_GENTLEMEN) >= 0 or _p1_which(value, _P1_TURNED) >= 0:
        return ("rose_row is %s, numbers from the gentlemen's table" % _p1_short(value),
                "%s's row is in ladies_matrix: ladies_matrix[%d]." % (names[i], i))
    if _p1_is_number(value):
        return ("rose_row is %s, a single number" % _p1_short(value),
                "a row is a whole list of numbers, and ladies_matrix[%d] picks one. A second index, as in "
                "ladies_matrix[%d][0], would pick one number out of that row." % (i, i))
    if _p1_rows_of(value):
        return ("rose_row is a whole matrix",
                "pick one row out of it with an index in square brackets, such as ladies_matrix[0] for the first "
                "row.")
    return ("rose_row is %s, and %s's row is a list of %d numbers" % (_p1_short(value), names[i], len(matrix[i])),
            _P1_COUNTING["ladies_matrix"])


def _p1_column_problem(value):
    """None when value is Felix's column of gentlemen_transposed, else (what went wrong, hint), for task A."""
    j = _P1_ONE[1]
    matrix, names = _P1_TURNED, _P1_GENTLEMAN_NAMES
    columns = _p1_columns(matrix)
    if _p1_which(value, [columns[j]]) == 0:
        return None
    k = _p1_which(value, columns)
    if k >= 0:
        return ("felix_column is %s, the column of %s, column(gentlemen_transposed, %d)" % (_p1_short(value),
                                                                                           names[k], k),
                _P1_COUNTING["gentlemen_transposed"])
    k = _p1_which(value, matrix)
    if k >= 0:
        return ("felix_column is %s, every gentleman's answer about %s: row %d of gentlemen_transposed, which is "
                "column %d of gentlemen_matrix" % (_p1_short(value), _P1_QUESTIONS[k], k, k),
                "a gentleman's numbers stand in a column of gentlemen_transposed, the table turned: "
                "column(gentlemen_transposed, %d) gives %s's column." % (j, names[j]))
    if _p1_which(value, _P1_LADIES) >= 0 or _p1_which(value, _p1_columns(_P1_LADIES)) >= 0:
        return ("felix_column is %s, from ladies_matrix" % _p1_short(value),
                "%s's column is in gentlemen_transposed: column(gentlemen_transposed, %d)." % (names[j], j))
    if _p1_is_number(value):
        return ("felix_column is %s, a single number" % _p1_short(value),
                "a column is a whole list of numbers, one per question: column(gentlemen_transposed, %d) gives "
                "%s's column." % (j, names[j]))
    if _p1_rows_of(value):
        return ("felix_column is a whole matrix",
                "pick one column out of it with column(gentlemen_transposed, j), such as "
                "column(gentlemen_transposed, 0) for %s, the first gentleman." % names[0])
    return ("felix_column is %s, and %s's column is a list of %d numbers"
            % (_p1_short(value), names[j], len(matrix)), _P1_COUNTING["gentlemen_transposed"])


def _p1_check_one(lady_row, gentleman_column, compatibility):
    gaps = []
    for variable, value in (("rose_row", lady_row), ("felix_column", gentleman_column),
                            ("rose_and_felix", compatibility)):
        if value is Ellipsis:
            gaps.append(variable)
    if gaps:
        verb = "holds"
        if len(gaps) > 1:
            verb = "hold"
        print("⏳ Task A is not finished yet: %s still %s ... (Python calls it Ellipsis). Replace each ... with "
              "your code, run the cell, then run this check again." % (_p1_and(gaps), verb))
        return
    i, j = _P1_ONE
    problem = _p1_row_problem(lady_row)
    if problem is None:
        problem = _p1_column_problem(gentleman_column)
        if problem is not None and _p1_given_broken() == "column":
            print("⏳ column does not work any more. %s, then the cell of task A, then this check." % _P1_RUN_GIVEN)
            return
    if problem is None:
        expected = _p1_inner(_P1_LADIES[i], _p1_column(_P1_TURNED, j))
        if not _p1_is_number(compatibility):
            problem = ("rose_and_felix is %s, and a compatibility is one number" % _p1_short(compatibility),
                       "rose_and_felix = inner_product(rose_row, felix_column).")
        elif compatibility != expected:
            if _p1_given_broken() == "inner_product":
                print("⏳ inner_product does not work any more. %s, then the cell of task A, then this check."
                      % _P1_RUN_GIVEN)
                return
            problem = ("rose_and_felix is %s, expected %d" % (_p1_short(compatibility), expected),
                       "let inner_product do the work: rose_and_felix = inner_product(rose_row, felix_column).")
    if problem is None:
        print("✅ " + _P1_SUCCESS["one_compatibility"])
        return
    print("❌ %s." % problem[0])
    print("   💡 Hint: " + problem[1])


def check_one_compatibility(rose_row, felix_column, rose_and_felix):
    """Task A: Rose's row, Felix's column, and their compatibility."""
    try:
        _p1_check_one(rose_row, felix_column, rose_and_felix)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


def _p1_check_big(product):
    for name, mine in (("big_a", _P1_BIG_A), ("big_b", _P1_BIG_B)):
        if name in globals() and globals()[name] != mine:
            print("❌ %s is no longer the matrix of the game's bigger product. Run the cell with big_a and big_b "
                  "again, then the cells below it." % name)
            return
    problem = _p1_not_a_table(product)
    if product is None or product is Ellipsis or (problem is not None and problem[0] == "wait"):
        print("⏳ big_product is not ready yet: your matrix_multiply of task C is not finished. Get its check to "
              "✅, then run the cells of the bigger pair again.")
        return
    if product == _P1_BIG_PRODUCT:
        print("✅ " + _P1_SUCCESS["big_product"])
        return
    what, hint = _p1_explain_product(product, _P1_BIG_A, _P1_BIG_B, _P1_BIG_PRODUCT)
    print("❌ big_product %s." % what.replace("gave ", "is ", 1))
    print("   💡 Hint: " + hint + " The check of task C tells you more.")


def check_big_product(big_product):
    """The bigger pair: compares big_product with the bigger product of the game."""
    try:
        _p1_check_big(big_product)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 1 again, then this cell." % error)


print("✅ Setup of part 1 complete.")

## The Matchmaker's two tables

The clients' answers as Python lists, each mark turned into its number as in the game. A matrix is a list of
rows. `ladies_matrix` has one row per lady and one column per question, in the order of `questions`.
`gentlemen_matrix` is the gentlemen's table as they filled in the form: one row per gentleman, one column per
question. Python counts from 0: `ladies_matrix[0]` is Clara's row, `[2, -1, -2]`, and `gentlemen_matrix[0]` is Arthur's,
`[1, -2, 1]`. The website, the clients and their answers are invented for the lesson.

In [ ]:
# The Matchmaker's data, from part one of the game (invented for the lesson)
questions = ["Dancing", "Reading", "Long walks"]
lady_names = ["Clara", "Eliza", "Harriet", "Rose"]
gentleman_names = ["Arthur", "Felix", "George", "Oscar", "Walter"]

# The ladies' answers: one row per lady, one column per question
ladies_matrix = [
    [ 2, -1, -2],   # Clara
    [ 1, -1,  2],   # Eliza
    [-2,  1, -2],   # Harriet
    [-2,  2,  2],   # Rose
]

# The gentlemen's answers, as they filled in the form: one row per gentleman, one column per question
gentlemen_matrix = [
    [ 1, -2,  1],   # Arthur
    [-2,  2,  1],   # Felix
    [-2,  2, -2],   # George
    [ 2, -2, -2],   # Oscar
    [-1,  0,  2],   # Walter
]

In [ ]:
print("ladies_matrix, the ladies' answers:")
show_matrix(ladies_matrix, lady_names, questions)
print("gentlemen_matrix, the gentlemen's answers:")
show_matrix(gentlemen_matrix, gentleman_names, questions)

## The inner product, a column and transposing, given

You wrote `inner_product` yourself in the video store notebook: multiply two lists of numbers position by
position, then add up the products. Python keeps a matrix as a list of rows, so a second given function,
`column(matrix, j)`, collects column j into a list: the number at position j of every row, from the top down.
The third, `transpose(matrix)`, turns a matrix: each row becomes a column. Run their cell.

In [ ]:
# Given: the inner product of two lists of numbers, the function you wrote in the video store notebook
def inner_product(row_a, row_b):
    # row_a and row_b are two lists of numbers of the same length,
    # such as a lady's row and a gentleman's column.
    # Multiply them position by position, then add up the products.
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


# Given: column j of a matrix, as a list
def column(matrix, j):
    # matrix is a list of rows. Column j holds the number at position j of every row, from the top down.
    numbers = []
    for row in matrix:
        numbers.append(row[j])
    return numbers


# Given: a matrix transposed, each row becomes a column
def transpose(matrix):
    # Row i of matrix becomes column i of the result. So the result has one row for each
    # column of matrix: its row j holds column j of matrix.
    turned = []
    for j in range(len(matrix[0])):
        turned.append(column(matrix, j))
    return turned

## Transposing the gentlemen's table

The Matchmaker wants the compatibility of every lady with every gentleman, in one table. First it turns the
gentlemen's table: each gentleman's row becomes a column. This is called **transposing**. In
`gentlemen_transposed` each gentleman has a column, and each question a row, in the order of the columns of
`ladies_matrix`. So a lady's row and a gentleman's column hold their answers to the same three questions, in the
same order, and can be compared. Run the cell, and the next one, which compares Clara and Arthur.

In [ ]:
# The Matchmaker turns the gentlemen's table: each gentleman's row becomes a column
gentlemen_transposed = transpose(gentlemen_matrix)

print("Arthur's row of gentlemen_matrix:       ", gentlemen_matrix[0])
print("Arthur's column of gentlemen_transposed:", column(gentlemen_transposed, 0))
show_matrix(gentlemen_transposed, questions, gentleman_names)

In [ ]:
# Clara is row 0 of ladies_matrix, and Arthur is column 0 of gentlemen_transposed
print("Clara and Arthur:", inner_product(ladies_matrix[0], column(gentlemen_transposed, 0)))

**Reading the result.** 2, as in the game. Clara's row `[2, -1, -2]` and Arthur's column `[1, -2, 1]`: the products, question by
question, are 2, 2 and -2, and their sum is 2 + 2 - 2 = 2. Both like dancing and both dislike reading, which adds 2 and 2; Clara hates long walks and
Arthur likes them, which adds -2. In Python: `2*1 + (-1)*(-2) + (-2)*1 = 2 + 2 + (-2) = 2`.

## 🎯 Task A. One compatibility

How compatible are Rose and Felix? Take Rose's row out of `ladies_matrix` by its index and Felix's column out of
`gentlemen_transposed` with `column`, then let `inner_product` compute their compatibility. Remember that Python
counts from 0: Clara is lady 0, and Arthur is gentleman 0, the first column. In the cell below, replace each `...` with
your code and run it.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Python counts the rows of a matrix from 0: ladies_matrix[0] is the first row, Clara's.
# Rose is the fourth lady. Which number picks her row?
# Felix is the second gentleman, the second column of gentlemen_transposed. Which number picks his column?

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
# rose_row = ladies_matrix[3]
# felix_column = column(gentlemen_transposed, ...)      # the second gentleman: count 0, 1
# rose_and_felix = inner_product(rose_row, felix_column)

In [ ]:
# 🎯 YOUR TURN, task A: one compatibility, Rose and Felix
rose_row = ...         # 🎯 YOUR TURN: Rose's row of ladies_matrix, picked by its index
felix_column = ...     # 🎯 YOUR TURN: Felix's column of gentlemen_transposed (use column)
rose_and_felix = ...   # 🎯 YOUR TURN: the inner product of the row and the column

print("Rose's row:    ", rose_row)
print("Felix's column:", felix_column)
print("Rose and Felix:", rose_and_felix)

In [ ]:
check_one_compatibility(rose_row, felix_column, rose_and_felix)

## 🎯 Task B. One row

A lady's row of the table of compatibilities holds five numbers, one for each gentleman. Write
`compatibilities_for_lady(lady_row, gentlemen_transposed)`. It gets the row of one lady and the gentlemen's
matrix, transposed, and returns a list with her compatibility with every gentleman, in the order of the columns.
For Clara, `compatibilities_for_lady(ladies_matrix[0], gentlemen_transposed)` gives `[2, -8, -2, 10, -6]`.

The loop is given. The number of columns of a matrix is the length of one of its rows, so
`for j in range(len(gentlemen_transposed[0]))` runs once for each gentleman, with j equal to 0, 1, 2, 3 and 4, and
`gentleman_column = column(gentlemen_transposed, j)` is the column of gentleman j. A list grows with `append`:
after `compatibilities = []` and `compatibilities.append(2)`, `compatibilities` is `[2]`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop runs once for each gentleman: j is 0, then 1, 2, 3 and 4, and gentleman_column is his column.
# In each round, the compatibility is the inner product of lady_row with gentleman_column.
# Then put it at the end of the list: compatibilities.append(compatibility).

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for j in range(len(gentlemen_transposed[0])):
#         gentleman_column = column(gentlemen_transposed, j)
#         compatibility = inner_product(lady_row, gentleman_column)
#         ...                                   # put compatibility at the end of the list compatibilities

In [ ]:
# 🎯 YOUR TURN, task B: one lady's compatibility with every gentleman
def compatibilities_for_lady(lady_row, gentlemen_transposed):
    # lady_row is the row of one lady; gentlemen_transposed has one column per gentleman.
    # Return a list with her compatibility with every gentleman, in the order of the columns.
    compatibilities = []
    for j in range(len(gentlemen_transposed[0])):            # one round per column: one per gentleman
        gentleman_column = column(gentlemen_transposed, j)   # the column of gentleman number j
        compatibility = ...   # 🎯 YOUR TURN: her compatibility with gentleman j (use inner_product)
        ...                   # 🎯 YOUR TURN: put compatibility at the end of the list compatibilities
    return compatibilities

In [ ]:
check_compatibilities_for_lady(compatibilities_for_lady)

In [ ]:
# Clara's compatibilities, shown as a table with one row: a list that holds the list of her compatibilities
clara_compatibilities = compatibilities_for_lady(ladies_matrix[0], gentlemen_transposed)
show_matrix([clara_compatibilities], ["Clara"], gentleman_names)

## 🎯 Task C. The whole table

The table of compatibilities has one row per lady, and each row is what your `compatibilities_for_lady`
computes. So the whole table is a loop around your loop: for every row of the first matrix, its inner products
with every column of the second. That is **matrix multiplication**, as in the game's scene "Matrix
multiplication" and in every textbook. The columns of the first matrix match the rows of the second, and every
row of the first meets every column of the second in an inner product. The product has one row for each row of
the first matrix and one column for each column of the second: its entry in row i and column j is the inner
product of row i of the first with column j of the second.

Write `matrix_multiply(matrix_a, matrix_b)`. It returns the product as a list of rows. You may assume that the
columns of `matrix_a` match the rows of `matrix_b`. `matrix_multiply(ladies_matrix, gentlemen_transposed)` should
give the Matchmaker's table of compatibilities, 4 rows and 5 columns.

💭 **Think first.** How many inner products does the whole table take?

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Read matrix_a as the ladies and matrix_b as the gentlemen, transposed.
# Row i of the product is then lady i's compatibility with every gentleman:
# the inner products of matrix_a[i] with every column of matrix_b.
# In task B you wrote a function that computes exactly one such row.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# compatibilities_for_lady(lady_row, gentlemen_transposed) from task B makes one row of the product.
# Here the lady's row is matrix_a[i], and the gentlemen's matrix, transposed, is matrix_b.

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     for i in range(len(matrix_a)):
#         new_row = compatibilities_for_lady(matrix_a[i], ...)
#         product.append(new_row)
#
# You may also write the loop of task B again, inside this loop:
#         new_row = []
#         for j in range(len(matrix_b[0])):
#             new_row.append(inner_product(..., column(matrix_b, j)))
#         product.append(new_row)

In [ ]:
# 🎯 YOUR TURN, task C: the whole table, matrix multiplication
def matrix_multiply(matrix_a, matrix_b):
    # The columns of matrix_a match the rows of matrix_b. The product has one row for each row
    # of matrix_a and one column for each column of matrix_b: its row i holds the inner
    # products of matrix_a[i] with every column of matrix_b.
    product = []
    for i in range(len(matrix_a)):
        new_row = ...   # 🎯 YOUR TURN: row i of the product, matrix_a[i] with every column of matrix_b
        product.append(new_row)
    return product

In [ ]:
check_matrix_multiply(matrix_multiply)

In [ ]:
# The whole table of compatibilities: the ladies' matrix times the gentlemen's matrix, transposed
show_matrix(matrix_multiply(ladies_matrix, gentlemen_transposed), lady_names, gentleman_names)

**Reading the result.** Twenty inner products, one for each cell: 4 ladies times 5 gentlemen. Each lady has
one best match: Clara and Oscar (10), Eliza and Arthur (5), Harriet and George (10) and Rose and Felix (10). Walter is no lady's best match. The lowest number, -12, belongs to Rose and Oscar: they disagree on all
three questions, and the products -4, -4 and -4 add up. One cell is 0: for Eliza and Oscar the products 2, 2 and -4 cancel out. These are
the twenty compatibilities of the game, and your `matrix_multiply` computed them all from the two matrices.
Inside it, `compatibilities_for_lady` runs a loop over the gentlemen, so matrix multiplication is two loops, one
inside the other: every row of the first matrix with every column of the second.

## 🎯 Task D. When two matrices can be multiplied

An inner product pairs a row of the first matrix with a column of the second, position by position, so the two
must be equally long. A row of the first matrix holds one number per column of that matrix, and a column of the
second holds one number per row of that matrix. That is why two matrices can be multiplied only when the
columns of the first match the rows of the second: then each number in a row of the first has a partner in
every column of the second. The rows of the first and the columns of the second can be any number, as with 4
ladies and 5 gentlemen.

💭 **Think first.** Can `ladies_matrix` be multiplied with `gentlemen_matrix`, the gentlemen's table as they
filled it in, before transposing? And with `gentlemen_transposed`?

Write `can_multiply(matrix_a, matrix_b)`. It returns `True` when the number of columns of `matrix_a` equals the
number of rows of `matrix_b`, and `False` otherwise. The number of rows of a matrix is its length,
`len(matrix_b)`, and its number of columns is the length of one of its rows, `len(matrix_a[0])`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# columns_a = len(matrix_a[0]) is given: the length of the first row of matrix_a.
# The number of rows of a matrix is its length: len(matrix_b) counts the rows of matrix_b.
# Then compare the two numbers with == inside an if.

In [ ]:
#@title 💡 Hint 2 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#     rows_b = len(matrix_b)
#     if columns_a == rows_b:
#         return True
#     else:
#         return ...

In [ ]:
# 🎯 YOUR TURN, task D: when two matrices can be multiplied
def can_multiply(matrix_a, matrix_b):
    # Return True when the columns of matrix_a match the rows of matrix_b, and False otherwise.
    columns_a = len(matrix_a[0])   # the number of columns of matrix_a: the length of its first row
    rows_b = ...                   # 🎯 YOUR TURN: the number of rows of matrix_b
    # 🎯 YOUR TURN: if columns_a and rows_b are equal, return True; else return False
    ...

In [ ]:
check_can_multiply(can_multiply)

In [ ]:
# The think-first question: the gentlemen's table as they filled it in, then transposed
print("Can ladies_matrix and gentlemen_matrix be multiplied?", can_multiply(ladies_matrix, gentlemen_matrix))
print("Can ladies_matrix and gentlemen_transposed be multiplied?", can_multiply(ladies_matrix, gentlemen_transposed))

**Reading the result.** `False`, then `True`. `ladies_matrix` has 3 columns, one per question, and
`gentlemen_matrix` has 5 rows, one per gentleman: a lady's row of 3 numbers would meet a column of 5.
Transposed, the gentlemen's table has 3 rows, one per question, so each number in a lady's row has a partner in
every column, the answer to the same question. That is why the Matchmaker transposes the gentlemen's table
first. The product has 4 rows, one for each lady, and 5 columns, one for each gentleman.

## A bigger product

The game's scene "A bigger product" multiplied a bigger pair, `big_a` with 5 rows and 4 columns and `big_b` with 4 rows and 6 columns: plain
invented numbers this time, with no ladies and no gentlemen. Without names, `show_matrix` numbers the rows and
the columns from 0, the way Python counts. The game numbered them from 1, so row 0 here is row 1 in the game.

In [ ]:
# The bigger pair of the game: plain invented numbers; big_a has 4 columns and big_b 4 rows
big_a = [
    [ 1, -2,  0,  2],
    [ 2,  1, -1,  0],
    [-1,  0,  2,  1],
    [ 0,  2,  1, -2],
    [ 2, -1,  1,  1],
]

big_b = [
    [ 1,  0,  2, -1,  1,  0],
    [ 1, -2,  0,  1,  2, -1],
    [ 0,  2, -1,  1, -2,  1],
    [-1,  1,  1,  0,  2, -2],
]

print("big_a:")
show_matrix(big_a)
print("big_b:")
show_matrix(big_b)

💭 **Think first.** Can `big_a` and `big_b` be multiplied? How many rows and columns will their product have? And
can `ladies_matrix` be multiplied with `big_b`? Then run the cell, and the check below it.

In [ ]:
print("Can big_a and big_b be multiplied?", can_multiply(big_a, big_b))
print("Can ladies_matrix and big_b be multiplied?", can_multiply(ladies_matrix, big_b))

big_product = matrix_multiply(big_a, big_b)
show_matrix(big_product)

In [ ]:
check_big_product(big_product)

**Reading the result.** 5 rows, one for each row of `big_a`, and 6 columns, one for each column of `big_b`: 30
inner products of 4 products each, the same table as in the game. The entry in row 2 and column 3 (row 3 and
column 4 in the game) is the inner product of `big_a[2]` and `column(big_b, 3)`: `(-1)*(-1) + 0*1 + 2*1 + 1*0 = 1 + 0 + 2 + 0 = 3`. `ladies_matrix` and
`big_b` cannot be multiplied: `ladies_matrix` has 3 columns and `big_b` has 4 rows.

## Back to the game

That is part 1: your `matrix_multiply` fills the Matchmaker's whole table of compatibilities from the clients'
answers. Part two of the game, Once Upon a Streaming Service, matrix factorization, starts from the other end.
Nobody fills in a form at a streaming service: it sees only ratings and looks for the numbers behind them. That
game sends you to part 2 of this notebook.

# Part 2. The predictions and the losses

In part two of the game the streaming service knows only the ratings: nobody fills in a form. It guesses an
embedding for every customer and every film, two numbers each, and measures how good the guess is: the
predictions, the loss of every rating, and their total. In this part you teach the computer the same
measurement:

- **Task A**: the predictions of a guess, with your `matrix_multiply` from part 1;
- **Task B**: the loss of one rating;
- **Task C**: the loss of every rating;
- **Task D**: the total loss, the whole guess in one number.

Run the setup cell of part 2 first.

If you come back after a break, Colab may have forgotten part 1. Then click on the setup cell below and choose
**Runtime > Run before** in Colab's menu: it runs every cell above it, part 1 included. Then run the setup
cell.

In [ ]:
#@title Setup of part 2 (run this first) { display-mode: "form" }
# The plumbing of part 2: print_loss, which prints one loss in a line, and the checks of tasks A to D.
# You do not need to read it. A check prints one short message and never stops the notebook.

# Part 2 uses three things from part 1: inner_product, matrix_multiply and show_matrix. When part 1 has not run in
# this session, this cell defines them, so that part 2 works on its own.
_p2_missing = []
if "inner_product" not in globals():
    _p2_missing.append("inner_product")

    def inner_product(row_a, row_b):
        # row_a and row_b are two lists of numbers of the same length.
        # Multiply them position by position, then add up the products.
        total = 0
        for i in range(len(row_a)):
            total = total + row_a[i] * row_b[i]
        return total

if "matrix_multiply" not in globals():
    _p2_missing.append("matrix_multiply")

    def matrix_multiply(matrix_a, matrix_b):
        # Every row of matrix_a with every column of matrix_b: row i of the product holds the inner
        # products of matrix_a[i] with each column of matrix_b.
        product = []
        for i in range(len(matrix_a)):
            new_row = []
            for j in range(len(matrix_b[0])):
                column_j = []
                for row in matrix_b:
                    column_j.append(row[j])
                new_row.append(inner_product(matrix_a[i], column_j))
            product.append(new_row)
        return product

if "show_matrix" not in globals():
    _p2_missing.append("show_matrix")

    def show_matrix(matrix, row_names=None, column_names=None):
        """A plain stand-in for the show_matrix of part 1: the table as text, one line per row."""
        try:
            problem = _p2_not_a_table(matrix)
            if problem is not None:
                print("⏳ Nothing to show yet: %s. Finish the 🎯 cell above, run it, then run this cell again."
                      % problem)
                return
            rows, columns = len(matrix), len(matrix[0])
            if not isinstance(row_names, (list, tuple)) or len(row_names) != rows:
                row_names = [str(i) for i in range(rows)]
            if not isinstance(column_names, (list, tuple)) or len(column_names) != columns:
                column_names = [str(j) for j in range(columns)]
            entries = [[_p2_text(matrix[i][j]) for j in range(columns)] for i in range(rows)]
            width = max(len(str(name)) for name in row_names)
            widths = []
            for j in range(columns):
                w = len(str(column_names[j]))
                for i in range(rows):
                    w = max(w, len(entries[i][j]))
                widths.append(w)
            print(" " * width + "".join("   " + str(column_names[j]).rjust(widths[j]) for j in range(columns)))
            for i in range(rows):
                print(str(row_names[i]).ljust(width)
                      + "".join("   " + entries[i][j].rjust(widths[j]) for j in range(columns)))
        except Exception as error:
            print("The table could not be shown (%s)." % error)


def _p2_short(value):
    """A value as text, cut short when it is long; a gap as words."""
    if value is Ellipsis:
        return "... (a gap in a 🎯 cell)"
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 80:
        text = text[:77] + "..."
    return text


def _p2_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _p2_text(x):
    """A number as text: 25, or at most 4 decimals: 10.6667, 0.25."""
    if not _p2_is_number(x):
        return _p2_short(x)
    if isinstance(x, float) and x == int(x) and abs(x) < 1e15:
        return "%d" % x
    if isinstance(x, float):
        if x != 0 and abs(x) < 0.0001:
            return "%.2g" % x
        return ("%.4f" % x).rstrip("0").rstrip(".")
    return str(x)


def _p2_rows(table):
    """A table as text: [[-3, 4], [2, 4], [2, -6]]."""
    try:
        return "[" + ", ".join("[" + ", ".join(_p2_text(x) for x in row) + "]" for row in table) + "]"
    except Exception:
        return _p2_short(table)


def _p2_waiting(value):
    """Why value is not a number yet, in words."""
    if value is Ellipsis:
        return "it is still ..., a gap in a 🎯 cell"
    if value is None:
        return "it is None (does the function end with a return line?)"
    return "it is %s" % _p2_short(value)


def _p2_not_a_table(matrix):
    """None when matrix is a table of numbers (rows of the same length), else the reason in words."""
    if matrix is Ellipsis:
        return "the table is still ..., a gap in a 🎯 cell"
    if matrix is None:
        return "there is no table yet, only None (does the function end with a return line?)"
    if not isinstance(matrix, (list, tuple)) or len(matrix) == 0:
        return "%s is not a table of numbers" % _p2_short(matrix)
    for row in matrix:
        if row is Ellipsis:
            return "a row is still ..., a gap in a 🎯 cell"
        if not isinstance(row, (list, tuple)):
            return "a row is %s, and each row of a table is a list of numbers" % _p2_short(row)
        if len(row) == 0:
            return "its rows are empty lists, []"
        for x in row:
            if x is Ellipsis:
                return "it still holds a ..., a gap in a 🎯 cell"
            if not _p2_is_number(x):
                return "it holds %s where a number belongs" % _p2_short(x)
    if len(set(len(row) for row in matrix)) > 1:
        return "its rows have different lengths"
    return None


# print_loss: one loss in a line

def print_loss(text, loss):
    """Print the text and a loss, at most 4 decimals, for example
    The total loss of the first guess: 64
    While a 🎯 task is not finished it prints a calm line instead."""
    try:
        if not _p2_is_number(loss):
            print("%s: ⏳ no loss yet, %s. The check of the 🎯 task above says more." % (text, _p2_waiting(loss)))
            return
        print("%s: %s" % (text, _p2_text(loss)))
    except Exception:
        pass


# The arithmetic, kept here for the checks

def _p2_inner(row_a, row_b):
    total = 0
    for i in range(len(row_a)):
        total = total + row_a[i] * row_b[i]
    return total


def _p2_column(matrix, j):
    numbers = []
    for row in matrix:
        numbers.append(row[j])
    return numbers


def _p2_product(a, b):
    """Every row of a with every column of b, as in part 1."""
    return [[_p2_inner(r, _p2_column(b, j)) for j in range(len(b[0]))] for r in a]


def _p2_loss(p, r):
    return (p - r) * (p - r)


def _p2_losses(predictions, ratings):
    return [[_p2_loss(predictions[i][j], ratings[i][j]) for j in range(len(ratings[i]))]
            for i in range(len(ratings))]


def _p2_flat_losses(predictions, ratings):
    out = []
    for row in _p2_losses(predictions, ratings):
        out.extend(row)
    return out


def _p2_total(table):
    total = 0
    for row in table:
        for x in row:
            total = total + x
    return total


# The checks

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer.
_P2_TESTS = {
    "predict": [
        (([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]]), [[-3, 4], [2, 4], [2, -6]],
         'predict([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]])',
         "The streaming service's first guess: Ida's row [-1, -2] with the column of Paddington, [-1, 2], gives 1 + (-4) = -3, and the same for the other five pairs"),
        (([[2]], [[3]]), [[6]],
         'predict([[2]], [[3]])',
         'One customer and one film with one number each: 2 times 3 is 6'),
        (([[1, 2]], [[3, 5], [4, 6]]), [[11, 17]],
         'predict([[1, 2]], [[3, 5], [4, 6]])',
         'One customer and two films, the columns [3, 4] and [5, 6]: one row with two predictions, 3 + 8 and 5 + 12'),
        (([[1, 0], [0, 1]], [[2], [3]]), [[2], [3]],
         'predict([[1, 0], [0, 1]], [[2], [3]])',
         'Two customers and one film, one column: a prediction for each customer'),
        (([[2, -1, -2]], [[1, 2], [-2, -2], [1, -2]]), [[2, 10]],
         'predict([[2, -1, -2]], [[1, 2], [-2, -2], [1, -2]])',
         "Three numbers per row and per column: Clara's row of part 1 with the columns of Arthur and Oscar, 2 + 2 - 2 = 2 and 4 + 2 + 4 = 10"),
        (([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]]), [[2, 1], [-1, 2], [1, -2]],
         'predict([[2, 1], [-1, 2], [1, -2]], [[1, 0], [0, 1]])',
         "A guess whose predictions are the ratings: the films' columns [1, 0] and [0, 1] pick each customer's first and second number"),
        (([[0.5, -1]], [[2], [0.5]]), [[0.5]],
         'predict([[0.5, -1]], [[2], [0.5]])',
         'Decimals: 0.5 times 2 plus -1 times 0.5 is 1 - 0.5 = 0.5'),
    ],
    "squared_loss": [
        ((-3, 2), 25,
         'squared_loss(-3, 2)',
         "The first guess's prediction for Ida and Paddington, -3, and her rating, 2, differ by -5, and -5 times -5 is 25"),
        ((2, 2), 0,
         'squared_loss(2, 2)',
         'A prediction that matches the rating misses by 0, and its loss is 0'),
        ((0, 2), 4,
         'squared_loss(0, 2)',
         'A prediction 2 too low: the difference is -2, and -2 times -2 is 4'),
        ((4, 2), 4,
         'squared_loss(4, 2)',
         'A prediction 2 too high: the difference is 2, and 2 times 2 is 4'),
        ((2, -1), 9,
         'squared_loss(2, -1)',
         'The prediction 2 and the rating -1 differ by 3, and 3 times 3 is 9'),
        ((1, 1), 0,
         'squared_loss(1, 1)',
         'The prediction 1 and the rating 1: no miss, no loss'),
        ((1.5, 2), 0.25,
         'squared_loss(1.5, 2)',
         'A prediction with decimals: the difference is -0.5, and -0.5 times -0.5 is 0.25'),
    ],
    "loss_matrix": [
        (([[-3, 4], [2, 4], [2, -6]], [[2, 1], [-1, 2], [1, -2]]), [[25, 9], [9, 4], [1, 16]],
         'loss_matrix([[-3, 4], [2, 4], [2, -6]], [[2, 1], [-1, 2], [1, -2]])',
         'The first guess: the prediction for Ida and Paddington, -3, and her rating, 2, differ by -5, and -5 times -5 is 25; the same for the other five ratings'),
        (([[2, 1], [-1, 2], [1, -2]], [[2, 1], [-1, 2], [1, -2]]), [[0, 0], [0, 0], [0, 0]],
         'loss_matrix([[2, 1], [-1, 2], [1, -2]], [[2, 1], [-1, 2], [1, -2]])',
         'Predictions equal to the ratings miss by 0, so every loss is 0'),
        (([[3]], [[1]]), [[4]],
         'loss_matrix([[3]], [[1]])',
         'One prediction, 3, and one rating, 1: the difference 2, and 2 times 2 is 4'),
        (([[0, 0, 0]], [[2, -1, 0]]), [[4, 1, 0]],
         'loss_matrix([[0, 0, 0]], [[2, -1, 0]])',
         'One customer, three films, every prediction 0: the losses 4, 1 and 0'),
        (([[0], [0]], [[1], [2]]), [[1], [4]],
         'loss_matrix([[0], [0]], [[1], [2]])',
         'Two customers and one film: the losses 1 and 4, one in each row'),
        (([[1.5, 1.25]], [[2, 1]]), [[0.25, 0.0625]],
         'loss_matrix([[1.5, 1.25]], [[2, 1]])',
         'Decimals: the differences -0.5 and 0.25, the losses 0.25 and 0.0625'),
    ],
    "total": [
        (([[25, 9], [9, 4], [1, 16]],), 64,
         'total([[25, 9], [9, 4], [1, 16]])',
         'The six losses of the first guess: 25 + 9 + 9 + 4 + 1 + 16 = 64'),
        (([[4]],), 4,
         'total([[4]])',
         'One loss alone: the total is the loss itself'),
        (([[1, 2, 3]],), 6,
         'total([[1, 2, 3]])',
         'One row of three losses: 1 + 2 + 3 = 6'),
        (([[1], [2], [6]],), 9,
         'total([[1], [2], [6]])',
         'One column of three losses: 1 + 2 + 6 = 9'),
        (([[0, 0], [0, 0], [0, 0]],), 0,
         'total([[0, 0], [0, 0], [0, 0]])',
         'Six losses of 0: the total is 0'),
        (([[1, 2], [3, 4], [5, 6], [7, 8]],), 36,
         'total([[1, 2], [3, 4], [5, 6], [7, 8]])',
         'Four rows of two losses: 1 + 2 + ... + 8 = 36'),
        (([[0.25, 0.5]],), 0.75,
         'total([[0.25, 0.5]])',
         'Decimals: 0.25 + 0.5 = 0.75'),
    ],
}

_P2_SUCCESS = {'predict': 'The first guess gives the six predictions of the board: -3 and 4 for Ida, 2 and 4 for Raj, 2 and -6 for Elsa.', 'squared_loss': "The prediction -3 and Ida's rating 2 differ by -5, and the loss is 25.", 'loss_matrix': 'The first guess gives the six losses of the board: 25 and 9 for Ida, 9 and 4 for Raj, 1 and 16 for Elsa.', 'total': 'The six losses of the first guess add up to 64, the total loss on the board.'}
_P2_FIRST_PREDICTIONS = [[-3, 4], [2, 4], [2, -6]]


def _p2_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced. Gives back (answer,
    error, the copies after the call); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    copies = copy.deepcopy(inputs)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copies)
        return answer, None, copies
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error), copies


def _p2_same(answer, expected):
    """True when answer is expected: a number within a tiny margin, or a table of such numbers."""
    try:
        if _p2_is_number(expected):
            return _p2_is_number(answer) and abs(answer - expected) < 1e-9
        if isinstance(expected, (list, tuple)):
            if not isinstance(answer, (list, tuple)) or len(answer) != len(expected):
                return False
            for a, e in zip(answer, expected):
                if not _p2_same(a, e):
                    return False
            return True
        return bool(answer == expected)
    except Exception:
        return False


def _p2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only
    # visible in the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _p2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _p2_flat(value):
    """The numbers inside a value of nested lists and tuples."""
    if isinstance(value, (list, tuple)):
        out = []
        for x in value:
            out.extend(_p2_flat(x))
        return out
    return [value]


def _p2_slip(fn, name, slips):
    """The hint of the first slip that fn repeats on every whole-number test of name, or None. A slip that stops
    with an error on a test matches only when fn stops with an error there too."""
    for slip, hint in slips:
        same = True
        tested = 0
        for inputs, expected, call, why in _P2_TESTS[name]:
            if not all(_p2_is_number(x) and x == int(x) for x in _p2_flat(inputs)):
                continue
            answer, error, after = _p2_call(fn, inputs)
            try:
                wanted, raises = slip(*inputs), False
            except Exception:
                wanted, raises = None, True
            tested += 1
            if raises:
                if error is None:
                    same = False
                    break
            elif error is not None or not _p2_same(answer, wanted):
                same = False
                break
        if same and tested > 0:
            return hint
    return None


def _p2_error_hint(error, name):
    """A hint for an error that stopped a participant's function."""
    if "NameError" in error:
        return ("Python does not know a name that %s uses. Run the setup cell of part 2 and the cells of the tasks "
                "above, then the cell of %s, then this check." % (name, name))
    if "unsupported operand" in error and "^" in error:
        return "in Python ^ means something else than a power. Multiply the difference by itself: difference * difference."
    if name == "predict":
        return ("predict gives its two matrices to matrix_multiply, the customers first: "
                "matrix_multiply(customer_embeddings, film_embeddings).")
    if name == "squared_loss":
        return ("squared_loss gets two numbers: difference = prediction - rating, then loss = difference * "
                "difference, then return loss.")
    if name == "loss_matrix":
        if "IndexError" in error:
            return ("predictions[i][j] and ratings[i][j] use the same i and j, in this order: i is the row, a "
                    "customer, and j the column, a film.")
        return ("in row i and column j: loss = squared_loss(predictions[i][j], ratings[i][j]), then row.append(loss) "
                "puts it at the end of the row.")
    if name == "total":
        if "ZeroDivisionError" in error:
            return ("the total loss divides by nothing: add every loss, running_total = running_total + "
                    "losses[i][j], then return running_total.")
        if "TypeError" in error and "list" in error:
            return ("losses[i] is a whole row, a list. One loss is losses[i][j]: running_total = running_total + "
                    "losses[i][j].")
        return "inside the loops, running_total = running_total + losses[i][j]; after them, return running_total."
    return "run the setup cell of part 2 again, then the cell of your function, then this check."


# Task A, predict

def _p2_their_product(a, b):
    """a times b by this session's matrix_multiply when it works (part 1's or the setup's), else by the rule of
    part 1. A slip such as the films first then gives what the participant's predict gives, however matrix_multiply
    was written, an error included."""
    fn = globals().get("matrix_multiply")
    if callable(fn) and _p2_works("matrix_multiply"):
        answer, error, after = _p2_call(fn, (a, b))
        if error is not None:
            raise ValueError(error)
        return answer
    return _p2_product(a, b)


_P2_PREDICT_SLIPS = [
    (lambda c, f: _p2_their_product(f, c),
     "this puts the films first. The customers go first, so that the predictions have one row per customer and "
     "one column per film: matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: [[_p2_inner(r, s) for s in f] for r in c],
     "this pairs every customer's row with every row of film_embeddings, and a row of film_embeddings holds one "
     "number of every film. A film's numbers stand in a column, and matrix_multiply pairs every row of the first "
     "matrix with every column of the second: matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _p2_their_product(c, c),
     "this multiplies the customers' matrix with itself. The second matrix is the films': "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _p2_their_product(f, f),
     "this multiplies the films' matrix with itself. The predictions pair the customers with the films: "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: c,
     "this is the customers' matrix itself. The predictions are its product with the films' matrix: "
     "matrix_multiply(customer_embeddings, film_embeddings)."),
    (lambda c, f: _P2_FIRST_PREDICTIONS,
     "the predictions must come from the two matrices that the function gets. Your code gave the predictions of "
     "the first guess every time: matrix_multiply(customer_embeddings, film_embeddings)."),
]


def _p2_run_predict(fn):
    """None when predict passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["predict"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "predict"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return predictions?" % call,
                    "the last line is return predictions.")
        if _p2_is_number(answer):
            return ("%s gave one number, %s, and the predictions are a table, %s. %s." % (
                call, _p2_text(answer), _p2_rows(expected), why),
                "one number is one prediction. predict returns the table of all of them, one row per customer: "
                "matrix_multiply(customer_embeddings, film_embeddings).")
        hint = _p2_slip(fn, "predict", _P2_PREDICT_SLIPS)
        if hint is None:
            hint = ("the predictions are the product of the two matrices, the customers first: "
                    "matrix_multiply(customer_embeddings, film_embeddings).")
        return ("%s gave %s, expected %s. %s." % (call, _p2_rows(answer), _p2_rows(expected), why), hint)
    return None


# Task B, squared_loss

_P2_LOSS_SLIPS = [
    (lambda p, r: p - r, "this is the difference, the prediction minus the rating. The loss is the difference "
                         "times itself: loss = difference * difference."),
    (lambda p, r: r - p, "this is the rating minus the prediction, and it is not squared yet. Take the prediction "
                         "minus the rating, then multiply that difference by itself."),
    (lambda p, r: 2 * (p - r), "times 2 doubles the difference. The loss multiplies the difference by itself: "
                               "difference * difference."),
    (lambda p, r: abs(p - r), "this is the size of the miss. The loss is the difference times itself: "
                              "difference * difference."),
    (lambda p, r: (p - r) ^ 2, "in Python ^ means something else than a power. Multiply the difference by "
                               "itself: difference * difference."),
    (lambda p, r: (p + r) * (p + r), "the difference is the prediction minus the rating: prediction - rating."),
    (lambda p, r: p * p, "this is the prediction times itself. Square the difference: difference = prediction - "
                         "rating, then difference * difference."),
    (lambda p, r: p * p - r * r, "square the difference itself: first difference = prediction - rating, then "
                                 "loss = difference * difference."),
]


def _p2_run_squared_loss(fn):
    """None when squared_loss passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["squared_loss"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "squared_loss"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return loss?" % call,
                    "the last line is return loss.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number, the loss: the difference times itself.")
        hint = _p2_slip(fn, "squared_loss", _P2_LOSS_SLIPS)
        if hint is None:
            hint = "difference = prediction - rating, then loss = difference * difference, then return loss."
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


# Task C, loss_matrix

_P2_LOSSES_SLIPS = [
    (lambda p, r: _p2_losses(r, r),
     "this compares a number with itself, so every loss is 0. Compare the prediction with the rating: "
     "squared_loss(predictions[i][j], ratings[i][j])."),
    (lambda p, r: [[p[i][j] - r[i][j] for j in range(len(p[i]))] for i in range(len(p))],
     "these are the differences, the predictions minus the ratings. The loss is the square of a difference: "
     "squared_loss(predictions[i][j], ratings[i][j])."),
    (lambda p, r: [[r[i][j] - p[i][j] for j in range(len(p[i]))] for i in range(len(p))],
     "these are the differences, the ratings minus the predictions. The loss is the square of a difference: "
     "squared_loss(predictions[i][j], ratings[i][j])."),
    (lambda p, r: _p2_flat_losses(p, r),
     "this is one flat list of all the losses. The losses are a table with one row per customer: each loss goes "
     "into row, row.append(loss), and the loop over i puts each finished row into losses."),
    (lambda p, r: [_p2_flat_losses(p, r) for i in range(len(p))],
     "every row holds all the losses so far. Start a new row for each customer: row = [] inside the loop over i, "
     "just before the loop over j."),
    (lambda p, r: [[] for i in range(len(p))],
     "the rows are empty: put each loss at the end of its row, row.append(loss)."),
]


def _p2_run_loss_matrix(fn):
    """None when loss_matrix passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["loss_matrix"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "loss_matrix"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with return losses?" % call,
                    "the last line is return losses.")
        hint = _p2_slip(fn, "loss_matrix", _P2_LOSSES_SLIPS)
        if hint is None:
            hint = ("in row i and column j: loss = squared_loss(predictions[i][j], ratings[i][j]), then "
                    "row.append(loss).")
        return ("%s gave %s, expected %s. %s." % (call, _p2_short(answer), _p2_rows(expected), why), hint)
    return None


# Task D, total. A version that divides the sum by a count gets the first hints.

_P2_TOTAL_SLIPS = [
    (lambda t: _p2_total(t) / sum(len(row) for row in t),
     "this divides the sum by the number of losses, which gives their average. The total loss is the sum itself: "
     "return running_total, with no division."),
    (lambda t: _p2_total(t) / len(t),
     "this divides the sum by len(losses), the number of rows. The total loss is the sum itself: return "
     "running_total, with no division."),
    (lambda t: _p2_total(t) / len(t[0]),
     "this divides the sum by len(losses[0]), the number of columns. The total loss is the sum itself: return "
     "running_total, with no division."),
    (lambda t: t[-1][-1],
     "running_total = losses[i][j] puts each loss in the place of the one before, so only the last loss stays. Add "
     "it to what is there: running_total = running_total + losses[i][j]."),
    (lambda t: sum(len(row) for row in t),
     "this counts the losses. Add the losses themselves: running_total = running_total + losses[i][j]."),
    (lambda t: _p2_total([t[0]]),
     "the return line runs inside the loop over the rows, so the total stops after the first row. Move "
     "return running_total out of both loops, as far left as running_total = 0."),
    (lambda t: t[0][0],
     "the return line runs inside the loop over the losses of a row, so the function stops at the first loss. Move "
     "return running_total out of both loops, as far left as running_total = 0."),
    (lambda t: _p2_total([[x * x for x in row] for row in t]),
     "the losses are squares already: loss_matrix squared each difference. Add them as they are: running_total = "
     "running_total + losses[i][j]."),
]


def _p2_run_total(fn):
    """None when total passes every test, else (what went wrong, hint)."""
    for inputs, expected, call, why in _P2_TESTS["total"]:
        answer, error, after = _p2_call(fn, inputs)
        if error is None and _p2_same(answer, expected):
            continue
        if error is not None:
            return ("%s stopped with an error: %s" % (call, error), _p2_error_hint(error, "total"))
        if answer is None:
            return ("%s gave no answer (None). Does the function end with a return line?" % call,
                    "the last line returns the sum of the losses: return running_total.")
        if not _p2_is_number(answer):
            return ("%s gave %s, expected one number, %s. %s." % (call, _p2_short(answer), _p2_text(expected), why),
                    "return one number: the sum of all the losses.")
        hint = _p2_slip(fn, "total", _P2_TOTAL_SLIPS)
        if hint is None:
            hint = "add every loss to running_total, then return running_total after the two loops."
        return ("%s gave %s, expected %s. %s." % (call, _p2_text(answer), _p2_text(expected), why), hint)
    return None


_P2_RUNNERS = {"predict": _p2_run_predict, "squared_loss": _p2_run_squared_loss,
               "loss_matrix": _p2_run_loss_matrix, "total": _p2_run_total}
_P2_NEEDS = {"predict": [("matrix_multiply", "part 1")],
             "squared_loss": [],
             "loss_matrix": [("squared_loss", "task B")],
             "total": []}


def _p2_works(name):
    """True when the function called name exists and passes its tests."""
    fn = globals().get(name)
    if not callable(fn):
        return False
    if name == "matrix_multiply":
        for a, b in (([[1, 2]], [[3], [4]]), ([[1, 0], [0, 1], [2, 2]], [[2, 3, 4], [5, 6, 7]]),
                     ([[-1, -2], [-2, 0], [2, 2]], [[-1, -2], [2, -1]])):
            answer, error, after = _p2_call(fn, (a, b))
            if error is not None or not _p2_same(answer, _p2_product(a, b)):
                return False
        return True
    if _p2_has_gap(fn):
        return False
    return _P2_RUNNERS[name](fn) is None


def _p2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s. If %s is no "
              "longer a function, run the cell of its task again." % (name, name, name, name, name))
        return
    if _p2_has_gap(fn):
        print("⏳ %s is not finished yet: replace each ... (Python calls it Ellipsis) with your code, run its "
              "cell, then run this check again." % name)
        return
    for other, place in _P2_NEEDS[name]:
        if not _p2_uses(fn, other) or _p2_works(other):
            continue
        if other == "matrix_multiply":
            print("⏳ %s uses your matrix_multiply from part 1, which does not work yet. Get the check of task C of "
                  "part 1, check_matrix_multiply(matrix_multiply), to ✅ first, then run this check again." % name)
        else:
            print("⏳ %s uses your %s from %s, which does not work yet. Get the check of %s to ✅ first, then run "
                  "this check again." % (name, other, place, place))
        return
    failure = _P2_RUNNERS[name](fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_P2_TESTS[name]), _P2_SUCCESS[name]))
        return
    print("❌ " + failure[0])
    print("   💡 Hint: " + failure[1])


def check_predict(fn):
    """Task A: tests predict on seven pairs of matrices, from the first guess to one with decimals."""
    try:
        _p2_check("predict", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_squared_loss(fn):
    """Task B: tests squared_loss on seven pairs of a prediction and a rating."""
    try:
        _p2_check("squared_loss", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_loss_matrix(fn):
    """Task C: tests loss_matrix on six pairs of tables, from the first guess to one with decimals."""
    try:
        _p2_check("loss_matrix", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


def check_total(fn):
    """Task D: tests total on seven tables of losses, from the first guess to one with decimals."""
    try:
        _p2_check("total", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the setup cell of part 2 again, then this cell." % error)


if _p2_missing:
    print("Note: part 1 has not run in this session, so this cell defined %s for part 2. To run part 1 as well, "
          "click on this cell, choose Runtime > Run before in the menu, then run this cell again."
          % (", ".join(_p2_missing[:-1]) + " and " + _p2_missing[-1] if len(_p2_missing) > 1 else _p2_missing[0]))
print("✅ Setup of part 2 complete.")

## The ratings

Three customers, Ida, Raj and Elsa, rated two films, *Paddington* (2014) and *The Martian* (2015), with the five marks of the game turned into
numbers: two ticks 2, a tick 1, a shrug 0, a cross -1, two crosses -2. `ratings` has one row per customer and
one column per film. The customers and their ratings are invented for the lesson.

In [ ]:
# The ratings of part two of the game (invented for the lesson): one row per customer, one column per film
customer_names2 = ["Ida", "Raj", "Elsa"]
film_names2 = ["Paddington", "The Martian"]
ratings = [
    [ 2,  1],   # Ida
    [-1,  2],   # Raj
    [ 1, -2],   # Elsa
]

In [ ]:
show_matrix(ratings, customer_names2, film_names2)

## The first guess

The streaming service guesses two numbers for each customer and two for each film. Nobody knows what the two
numbers mean, so they have no names: `number_names` calls them the first and the second number. As the ladies
and the gentlemen in part 1, the customers' numbers stand in rows and the films' numbers in columns:
`customer_embeddings` has one row per customer, and `film_embeddings` has one row per number and one column per
film, as the board of the game draws the films above the predictions. The cell below holds the streaming
service's first guess from the board of the game.

In [ ]:
# The streaming service's first guess, as on the board of the game: two numbers per customer and per film
customer_embeddings = [
    [-1, -2],   # Ida
    [-2,  0],   # Raj
    [ 2,  2],   # Elsa
]
# The films: one row per number, one column per film, in the order of film_names2: Paddington, The Martian
film_embeddings = [
    [-1, -2],   # first number
    [ 2, -1],   # second number
]
number_names = ["first number", "second number"]
show_matrix(customer_embeddings, customer_names2, number_names)
show_matrix(film_embeddings, number_names, film_names2)

## 🎯 Task A. The predictions

The prediction of a customer's rating of a film is the inner product of the customer's row and the film's
column. All the predictions together are the product of the two matrices, every customer's row with every
film's column: the matrix multiplication of part 1. Like `ratings`, the predictions have one row per customer
and one column per film.

Write `predict(customer_embeddings, film_embeddings)`. It gets the two matrices and returns the predictions.

💭 **Think first.** Which of the two matrices goes first in the product?

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The predictions are the product of the two matrices, every customer's row with every film's column.
# Part 1's matrix_multiply(matrix_a, matrix_b) computes exactly that. The rows of matrix_a become
# the rows of the product, and the columns of matrix_b its columns.

In [ ]:
#@title 💡 Hint 2 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     predictions = matrix_multiply(..., ...)   # the customers first: one row of predictions per customer
#     return predictions

In [ ]:
# 🎯 YOUR TURN, task A: the predictions of a guess
def predict(customer_embeddings, film_embeddings):
    # customer_embeddings has one row per customer, film_embeddings one column per film. Return
    # the predictions: one row per customer, one column per film, each the inner product of a
    # customer's row and a film's column.
    predictions = ...   # 🎯 YOUR TURN: the product of the two matrices (use matrix_multiply)
    return predictions

In [ ]:
check_predict(predict)

In [ ]:
# The predictions of the first guess: one row per customer, one column per film
predictions = predict(customer_embeddings, film_embeddings)
show_matrix(predictions, customer_names2, film_names2)

**Reading the result.** The six predictions of the board: -3 and 4 for Ida, 2 and 4 for Raj, 2 and -6 for Elsa. Ida's row `[-1, -2]` and the column of *Paddington*, `[-1, 2]`:
the products are 1 and -4, and their sum is -3. Ida gave *Paddington* 2, so this prediction misses. The customers'
matrix goes first: its rows become the rows of the product, one per customer, and the columns of the films'
matrix become its columns, one per film.

## 🎯 Task B. The loss of one rating

How far off is a prediction? Its **loss** is the prediction minus the rating, squared, as in the game. For Ida
and *Paddington*: -3 - 2 = -5, and -5 times -5 is 25.

Write `squared_loss(prediction, rating)`. It gets two numbers, the prediction and the rating, in this order, and
returns the loss.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The difference is the prediction minus the rating: for Ida and Paddington, -3 - 2 = -5.
# In Python: difference = prediction - rating
# The loss is that difference times itself.

In [ ]:
#@title 💡 Hint 2 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     difference = prediction - rating
#     loss = ...                         # the difference times itself, with *
#     return loss

In [ ]:
# 🎯 YOUR TURN, task B: the loss of one rating
def squared_loss(prediction, rating):
    # prediction and rating are two numbers. Return their difference times itself.
    difference = ...   # 🎯 YOUR TURN: the prediction minus the rating
    loss = ...         # 🎯 YOUR TURN: the difference times itself
    return loss

In [ ]:
check_squared_loss(squared_loss)

In [ ]:
# The first guess's prediction for Ida and Paddington, -3, and three other predictions, against her rating, 2
print_loss("The loss of the prediction -3", squared_loss(-3, 2))
print_loss("The loss of the prediction 1", squared_loss(1, 2))
print_loss("The loss of the prediction 0", squared_loss(0, 2))
print_loss("The loss of the prediction 4", squared_loss(4, 2))

**Reading the result.** 25 for the first guess. A miss of 1 costs 1 and a miss of 2 costs 4: twice the miss,
four times the loss, so bigger mistakes are punished harder. The predictions 0 and 4 both miss by 2, one too
low and one too high, and both cost 4: -2 times -2 is 4, as is 2 times 2. Too high and too low count alike.

## 🎯 Task C. The loss of every rating

The board of the game shows a loss for every rating, in a table of the same shape as `ratings`. Write
`loss_matrix(predictions, ratings)`, the predictions first, as in task B. In row `i` and column `j` it puts the
loss of the prediction `predictions[i][j]` and the rating `ratings[i][j]`. The skeleton gives the two loops,
over the rows `i` and the columns `j`, as in part 1: you fill in the loss and put it at the end of its row.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# In row i and column j sit the prediction predictions[i][j] and the rating ratings[i][j].
# squared_loss from task B gives their loss. row.append(loss) puts it at the end of the row.

In [ ]:
#@title 💡 Hint 2 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#             loss = squared_loss(predictions[i][j], ...)
#             row.append(...)

In [ ]:
# 🎯 YOUR TURN, task C: the loss of every rating
def loss_matrix(predictions, ratings):
    # predictions and ratings are two tables of the same shape: one row per customer, one
    # column per film. Return the table of the losses, of the same shape: in row i and
    # column j, the squared loss of predictions[i][j] and ratings[i][j].
    losses = []
    for i in range(len(ratings)):            # row i: one customer
        row = []
        for j in range(len(ratings[i])):     # column j: one film
            loss = ...                        # 🎯 YOUR TURN: the loss in row i, column j (use squared_loss)
            ...                               # 🎯 YOUR TURN: put the loss at the end of row
        losses.append(row)
    return losses

In [ ]:
check_loss_matrix(loss_matrix)

In [ ]:
# The loss of every rating of the first guess
losses = loss_matrix(predictions, ratings)
show_matrix(losses, customer_names2, film_names2)

**Reading the result.** The six losses of the board: 25 and 9 for Ida, 9 and 4 for Raj, 1 and 16 for Elsa. The largest, 25, is Ida's rating of *Paddington*, the
prediction that misses most.

## 🎯 Task D. The total loss

Six losses are hard to compare from one guess to the next. Their **total**, the sum of all of them, measures
the whole guess in one number: the total loss. The smaller, the better; it is 0 only when every prediction
equals its rating.

Write `total(losses)`. It gets a table of losses and returns the sum of all its numbers.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Visit every loss and add it to running_total. After the two loops, running_total holds
# the sum of all the losses. For the first guess: 25 + 9 + 9 + 4 + 1 + 16 = 64.

In [ ]:
#@title 💡 Hint 2 for task D: the shape of the code (double-click to open) { display-mode: "form" }
#             running_total = running_total + losses[i][j]
#     return ...                      # the sum, after both loops

In [ ]:
# 🎯 YOUR TURN, task D: the total loss
def total(losses):
    # losses is a table, a list of rows of numbers. Return the sum of all its numbers.
    running_total = 0
    for i in range(len(losses)):
        for j in range(len(losses[i])):
            running_total = ...   # 🎯 YOUR TURN: the sum so far plus losses[i][j]
    return ...                    # 🎯 YOUR TURN: the sum of all the losses

In [ ]:
check_total(total)

In [ ]:
# The total loss of the first guess
total_loss = total(losses)
print_loss("The total loss of the first guess", total_loss)

**Reading the result.** 64: the six losses add up to 25 + 9 + 9 + 4 + 1 + 16 = 64, the total loss on the board of the game.
That one number measures the whole guess.

## Back to the game

That is part 2: your `predict`, `loss_matrix` and `total` measure a guess the way the board of the game does,
the measurement behind every small change in the game's rounds. Next, back in part two of the Streaming
Service: the whole picture. The optional game, gradient descent up close, goes with the optional notebook:
[the optional notebook in Colab](https://colab.research.google.com/github/eth-ainit-hs26/w2-cx-public/blob/main/02_cx_matrix_factorization_optional_student.ipynb).